# Implementing, Quantizing, and Saving the Weights of a Small Network

In this notebook, I collect and present the training and quantization of the network. As a first step, I implemented and quantized a small, fixed-size network.

## Implementing a Fixed-Size Network

Libraries used

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
import yaml
import matplotlib.cm as cm
from PIL import Image
from typing import Dict
import brevitas.nn as qnn
from brevitas.quant import Int8ActPerTensorFloat, Int8WeightPerTensorFloat, SignedTernaryWeightPerTensorConst, SignedTernaryActPerTensorConst, Int8BiasPerTensorFixedPointInternalScaling, Int8WeightPerTensorFixedPoint, Int8ActPerTensorFixedPoint, Int8BiasPerTensorFloatInternalScaling
from pathlib import Path

### Floating point model

The initial network with FP32

In [ ]:
class RGBClassifierNN(nn.Module):
    def __init__(self, HIDDEN_LAYER_1=4, HIDDEN_LAYER_2=4):
        super(RGBClassifierNN, self).__init__()
        self.fc1 = nn.Linear(3, HIDDEN_LAYER_1, bias=True)
        self.fc2 = nn.Linear(HIDDEN_LAYER_1, HIDDEN_LAYER_2, bias=True)
        self.fc3 = nn.Linear(HIDDEN_LAYER_2, 1, bias=True)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x
    
    def classify_image(self, image_path, output_path="output_heatmap.png", cmap_name="inferno"):

        self.eval()

        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        img_array = (img_array - 127.5) / 127.5

        height, width, _ = img_array.shape
        output_array = np.zeros((height, width), dtype=np.float32)

        with torch.no_grad():
            for i in range(height):
                for j in range(width):
                    pixel = torch.tensor(img_array[i, j], dtype=torch.float32).unsqueeze(0).to("cuda")
                    output = self.forward(pixel).item()
                    output_array[i, j] = output
        
        output_array = np.clip(output_array, 0.0, 1.0)

        cmap = cm.get_cmap(cmap_name)
        heatmap = cmap(output_array)[:, :, :3]

        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))
        heatmap_img.save(output_path)

        print(f"Hőtérkép elmentve ide: {output_path}")

        return output_array


Processing the images and defining the training set:

In [ ]:
def process_image(image_path, label, white_correction=True):
    image = np.array(Image.open(image_path).convert("RGB"))
    if white_correction:
        mask = np.all(image != [255, 255, 255], axis=-1)
    return np.column_stack((image[mask], np.full(mask.sum(), label, dtype=np.uint8)))

def prepare_data(image1_path, image2_path, white_correction=True, test_size=10000):
    data = np.vstack([process_image(image1_path, 0, white_correction)] + [process_image(image2_path, 1, white_correction)] * 80)
    np.random.shuffle(data)
    data = data.astype(np.float32)
    data = np.array(data, dtype=np.float32)

    X = (data[:, :3]) / 256.0
    y = data[:, 3].reshape(-1, 1)

    if test_size > len(X):
        test_size = len(X)

    X_remaining, X_test, y_remaining, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_remaining, y_remaining, test_size=0.2, random_state=42, stratify=y_remaining
    )

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32)

    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.float32)

    X_test = torch.tensor(X_test, dtype=torch.float32)
    y_test = torch.tensor(y_test, dtype=torch.float32)

    return X_train, y_train, X_val, y_val, X_test, y_test

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image1_path = "./files/street_B_sarga_nelkul.bmp"
image2_path = "./files/street_B_sarga.bmp"
image3_path = "./files/greg-jewett-bb7l7MB05_I-unsplash.bmp"
X_train, y_train, X_val, y_val, X_test, y_test = prepare_data(image1_path, image2_path)

Function required for training and testing:

In [ ]:
def display_loss_plot(losses, title="Training loss", xlabel="Iterations", ylabel="Loss"):
    x_axis = [i for i in range(len(losses))]
    plt.plot(x_axis,losses)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()


def load_config(config_path="config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def get_criterion(name: str):
    if name == "BCELoss":
        return nn.BCELoss()
    elif name == "BCEWithLogitsLoss":
        return nn.BCEWithLogitsLoss()
    elif name == "CrossEntropyLoss":
        return nn.CrossEntropyLoss()
    elif name == "MSELoss":
        return nn.MSELoss()
    else:
        raise ValueError(f"Unknown criterion: {name}")

def get_optimizer(name: str, model_params, lr, weight_decay=0.0, momentum=0.0):
    if name == "Adam":
        return optim.Adam(model_params, lr=lr, weight_decay=weight_decay)
    elif name == "AdamW":
        return optim.AdamW(model_params, lr=lr, weight_decay=weight_decay)
    elif name == "SGD":
        return optim.SGD(model_params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif name == "RMSprop":
        return optim.RMSprop(model_params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

def train_model(model, X_train, y_train, X_val, y_val, config_path="config.yaml"):
    config = load_config(config_path)
    params = config["learning_params"]

    # Load parameters
    learning_rate = params["learning_rate"]
    criterion = get_criterion(params["criterion"])
    optimizer = get_optimizer(params["optimizer"], model.parameters(),
                              lr=learning_rate,
                              weight_decay=params.get("weight_decay", 0.0),
                              momentum=params.get("momentum", 0.0))
    epochs = params["num_of_epochs"]
    patience = params.get("patience", 50)
    early_stopping_threshold = params.get("early_stopping_threshold", 0.001)
    device = torch.device(params.get("device", "cuda" if torch.cuda.is_available() else "cpu"))

    X_train = X_train.clone().detach().float().to(device)
    y_train = y_train.clone().detach().float().to(device)
    X_val = X_val.clone().detach().float().to(device)
    y_val = y_val.clone().detach().float().to(device)
    model.to(device)
    criterion = criterion.to(device)

    train_losses = []
    val_losses = []
    epochs_since_improvement = 0 
    best_val_loss = float("inf")

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)

        train_losses.append(loss.item())
        val_losses.append(val_loss.item())
        
        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Train Loss: {loss.item():.6f}, Val Loss: {val_loss.item():.6f}")
        if val_loss < best_val_loss - early_stopping_threshold:
            best_val_loss = val_loss
            epochs_since_improvement = 0
        else:
            epochs_since_improvement += 1

        if epochs_since_improvement >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break
    
    loss_per_epoch = [np.mean(loss_per_epoch) for loss_per_epoch in train_losses]
    display_loss_plot(loss_per_epoch)

    return model

def test(model: nn.Module, X: torch.Tensor, y: torch.Tensor, threshold: float = 0.5, total_iterations: int = None) -> Dict[str, float]:
    TP = TN = FP = FN = 0
    total = 0
    model.eval()
    
    with torch.no_grad():
        max_samples = len(X) if total_iterations is None else min(len(X), total_iterations)
        for i in range(max_samples):
            x_i = X[i].unsqueeze(0)
            y_i = y[i].item()
            output = model(x_i).item()
            pred = 1 if output >= threshold else 0
            
            if pred == 1 and y_i == 1:
                TP += 1
            elif pred == 0 and y_i == 0:
                TN += 1
            elif pred == 1 and y_i == 0:
                FP += 1
            elif pred == 0 and y_i == 1:
                FN += 1
                
            total += 1

            if i % 1000 == 0 and i > 0:
                print(f"Processed {i} samples...")
    
    accuracy = (TP + TN) / total if total > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    
    print(f'TP={TP}, TN={TN}, FP={FP}, FN={FN}')
    print(f'Accuracy: {round(accuracy, 8)}')
    print(f'Precision: {round(precision, 8)}')
    print(f'Recall: {round(recall, 8)}')
    print(f'F1-Score: {round(f1_score, 8)}')
    print(f'Specificity: {round(specificity, 8)}')
    
    results = {
        'accuracy': round(accuracy, 8),
        'precision': round(precision, 8),
        'recall': round(recall, 8),
        'f1_score': round(f1_score, 8),
        'specificity': round(specificity, 8),
        'confusion_matrix': {
            'TP': TP,
            'TN': TN, 
            'FP': FP,
            'FN': FN
        }
    }
    
    return results

Training:

In [ ]:
floating_point_model = RGBClassifierNN()
floating_point_model = train_model(floating_point_model, X_train, y_train, X_val, y_val)
report = test(floating_point_model.to("cpu"), X_test, y_test)

In [ ]:
floating_point_model.to("cuda")
floating_point_model.classify_image(image3_path, output_path="floating_point_heatmap.png", cmap_name="inferno")

### Quantized Network - in PyTorch

The implementation matches the floating-point model, with the difference that it places a layer at the beginning of the network that quantizes the input from FP32 to INT8, and another layer at the end that converts the output back from INT8 to FP32.

In [ ]:
class QuantRGBClassifierNN(nn.Module):    
    def __init__(self, HIDDEN_LAYER_1=4, HIDDEN_LAYER_2=4):
        super(QuantRGBClassifierNN, self).__init__()
        self.quant = torch.quantization.QuantStub()
        self.fc1 = nn.Linear(3, HIDDEN_LAYER_1, bias=True)
        self.fc2 = nn.Linear(HIDDEN_LAYER_1, HIDDEN_LAYER_2, bias=True)
        self.fc3 = nn.Linear(HIDDEN_LAYER_2, 1, bias=True)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        x = self.dequant(x)
        return x
    
    def classify_image(self, image_path, output_path="output_quant_heatmap.png", cmap_name="inferno"):

        self.eval()

        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        img_array = (img_array - 127.5) / 127.5 

        height, width, _ = img_array.shape
        output_array = np.zeros((height, width), dtype=np.float32)

        with torch.no_grad():
            for i in range(height):
                for j in range(width):
                    pixel = torch.tensor(img_array[i, j], dtype=torch.float32).unsqueeze(0).to("cuda")
                    output = self.forward(pixel).item()
                    output_array[i, j] = output

        output_array = np.clip(output_array, 0.0, 1.0)

        cmap = cm.get_cmap(cmap_name)
        heatmap = cmap(output_array)[:, :, :3]

        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))
        heatmap_img.save(output_path)

        print(f"Hőtérkép elmentve ide: {output_path}")

        return output_array

This is where the quantized network is instantiated and calibrated (using the test data), as well as tested.

In [ ]:
quant_model = QuantRGBClassifierNN().to(device="cpu")
quant_model.load_state_dict(floating_point_model.state_dict())
quant_model.eval()

quant_model.qconfig = torch.ao.quantization.default_qconfig
quant_model = torch.ao.quantization.prepare(quant_model, inplace=True)

# print(quant_model)
X_test.to("cuda")
y_test.to("cuda")
test(quant_model, X_test, y_test)
quant_model = torch.ao.quantization.convert(quant_model, inplace=True)
# print(quant_model)

### Brevitas PTQ Quantization

Defining the network is very similar to the previous ones, except that the weights and activations can be quantized with separate parameters.

In [ ]:
class QuantRGBClassifierNN_Brevitas(nn.Module):
    def __init__(self, HIDDEN_LAYER_1=4, HIDDEN_LAYER_2=4, act_quant=Int8ActPerTensorFloat, weight_quant=Int8WeightPerTensorFloat):
        super(QuantRGBClassifierNN_Brevitas, self).__init__()

        self.quant_input = qnn.QuantIdentity(
            act_quant=act_quant,
        )

        self.fc1 = qnn.QuantLinear(
            in_features=3,
            out_features=HIDDEN_LAYER_1,
            weight_quant=weight_quant,
            bias=True,
            bias_quant=Int8BiasPerTensorFloatInternalScaling,
        )
        
        self.fc2 = qnn.QuantLinear(
            in_features=HIDDEN_LAYER_1, 
            out_features=HIDDEN_LAYER_2,
            weight_quant=weight_quant,
            bias=True,
            bias_quant=Int8BiasPerTensorFloatInternalScaling,
        )
        
        self.fc3 = qnn.QuantLinear(
            in_features=HIDDEN_LAYER_2,
            out_features=1,
            weight_quant=weight_quant,
            bias=True,
            bias_quant=Int8BiasPerTensorFloatInternalScaling,
        )
        
        self.relu1 = qnn.QuantReLU(
            act_quant=act_quant,
        )   

        self.relu2 = qnn.QuantReLU(
            act_quant=act_quant,
        )

        self.sigmoid = qnn.QuantReLU(
            act_quant=act_quant,
        )
    
    def forward(self, x):
        x = self.quant_input(x)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x
    
    def classify_image(self, image_path, output_path="output_heatmap.png", cmap_name="inferno"):

        self.eval()

        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        img_array = (img_array - 127.5) / 127.5

        height, width, _ = img_array.shape
        output_array = np.zeros((height, width), dtype=np.float32)

        with torch.no_grad():
            for i in range(height):
                for j in range(width):
                    pixel = torch.tensor(img_array[i, j], dtype=torch.float32).unsqueeze(0).to("cuda")
                    output = self.forward(pixel).item()
                    output_array[i, j] = output
        
        # Optional
        output_array = np.clip(output_array, 0.0, 1.0)

        cmap = cm.get_cmap(cmap_name)
        heatmap = cmap(output_array)[:, :, :3]

        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))
        heatmap_img.save(output_path)

        print(f"Hőtérkép elmentve ide: {output_path}")

        return output_array

Instantiating and calibrating the Brevitas network.

In [ ]:
brevitas_model = QuantRGBClassifierNN_Brevitas().to(device="cpu")
brevitas_model.eval()
with torch.no_grad():
    brevitas_model.fc1.weight.data.copy_(floating_point_model.fc1.weight.data)
    brevitas_model.fc2.weight.data.copy_(floating_point_model.fc2.weight.data)
    brevitas_model.fc3.weight.data.copy_(floating_point_model.fc3.weight.data)
    brevitas_model.fc1.bias.data.copy_(floating_point_model.fc1.bias.data)
    brevitas_model.fc2.bias.data.copy_(floating_point_model.fc2.bias.data)
    brevitas_model.fc3.bias.data.copy_(floating_point_model.fc3.bias.data)
    
calibration_data = X_train[:10000] 
print(f"Kalibrációs adatok mérete: {calibration_data.shape}")

brevitas_model.train()
with torch.no_grad():
    for i in range(min(100, len(calibration_data))):
        sample = calibration_data[i].unsqueeze(0)
        _ = brevitas_model(sample)
        if i % 20 == 0:
            print(f"Kalibráció: {i}/100")
brevitas_model.eval()
print("PTQ kalibráció befejezve.")

### Quantization Evaluation

In [ ]:
print("Test floating point model:")
floating_point_model.to("cpu")

report = test(floating_point_model, X_test, y_test)
floating_point_model.to("cuda")
print('\n')

print("Test quantized model:")
report = test(quant_model, X_test, y_test)
print('\n')

print("Test Brevitas quantized model:")
report = test(brevitas_model, X_test, y_test)
print('\n')

Printing weights and different representations (for testing and comparing the results)

In [ ]:
print("FP32:")
print(floating_point_model.fc1.weight)
print(floating_point_model.fc2.weight)
print(floating_point_model.fc3.weight)
print("Bias")
print(floating_point_model.fc1.bias)
print(floating_point_model.fc2.bias)
print(floating_point_model.fc3.bias)
print("\n")
print("PyTorch quant FP32:")
print(quant_model.fc1.weight())
print(quant_model.fc2.weight())
print(quant_model.fc3.weight())
print("Bias")
print(quant_model.fc1.bias())
print(quant_model.fc2.bias())
print(quant_model.fc3.bias())
print("\n")
print("INT8 representation:")
print(quant_model.fc1.weight().int_repr())
print(quant_model.fc2.weight().int_repr())
print(quant_model.fc3.weight().int_repr())
#print("Bias")
#print(quant_model.fc1.bias().int_repr())
#print(quant_model.fc2.bias().int_repr())
#print(quant_model.fc3.bias().int_repr())
print("\n")
print("INT8 Brevitas:")
print(brevitas_model.fc1.quant_weight())
print(brevitas_model.fc2.quant_weight())
print(brevitas_model.fc3.quant_weight())
print("Bias")
print(brevitas_model.fc1.quant_bias())
print(brevitas_model.fc2.quant_bias())
print(brevitas_model.fc3.quant_bias())
print("\n")
"""
print("INT8 Brevitas representation:")
print(brevitas_model.fc1.int_weight())
print(brevitas_model.fc2.int_weight())
print(brevitas_model.fc3.int_weight())
print("Bias")
print(brevitas_model.fc1.int_bias())
print(brevitas_model.fc2.int_bias())
print(brevitas_model.fc3.int_bias())
"""
# Brevitas model - integer repres
print("INT8 Brevitas representation:")
print(brevitas_model.fc1.quant_weight().int())
print(brevitas_model.fc2.quant_weight().int())
print(brevitas_model.fc3.quant_weight().int())

Helper function for writing to a file. Parameters: list of layers, file name, model type (backend).

In [ ]:
def extract_params_hls(params, filename, backend="float"):
    with open(f"{filename}_weight.txt", "w") as f:
        for layer in params:
            if backend == "float":
                values = layer.detach().cpu().numpy()
                #bias = layer.bias.detach().cpu().numpy() if layer.bias is not None else None
            elif backend == "torch_quant":
                values = layer.weight().int_repr().cpu().numpy()
                #bias = layer.bias().int_repr().cpu().numpy() if layer.bias is not None else None
            elif backend == "brevitas":
                values = layer.quant_weight().int().cpu().numpy()
                #bias = layer.int_bias().cpu().numpy() if layer.bias is not None else None
            else:
                raise ValueError(f"Ismeretlen backend: {backend}")
            
            for row in values:
                for item in row:
                    f.write(f"{item}\n")

    with open(f"{filename}_bias.txt", "w") as f:
        for layer in params:
            if backend == "float":
                bias = layer.bias.detach().cpu().numpy() if layer.bias is not None else None
            elif backend == "torch_quant":
                bias = layer.bias().int_repr().cpu().numpy() if layer.bias is not None else None
            elif backend == "brevitas":
                bias = layer.quant_bias().int().cpu().numpy() if layer.bias is not None else None
            else:
                raise ValueError(f"Ismeretlen backend: {backend}")
            
            for item in bias:
                f.write(f"{item}\n")

    if backend == "brevitas":
        with open(f"{filename}_weight_scale.txt", "w") as f:
            for layer in params:
                f.write(f"{int(np.abs(np.log2(layer.quant_weight().scale.data.item())))}\n")
        
        with open(f"{filename}_bias_scale.txt", "w") as f:
            for layer in params:
                if layer.bias is not None:
                    f.write(f"{int(np.abs(np.log2(layer.quant_bias().scale.data.item())))}\n")
                else:
                    f.write("0\n")    

In [ ]:
extract_params_hls(
    [brevitas_model.fc1, brevitas_model.fc2, brevitas_model.fc3],
    "brevitas_params", backend="brevitas"
)

## Implementing Configurable Networks

As a next step, I implemented configurable networks. The configuration can be modified in a 'confog.yaml' file, which looks something like this:
```yaml
network:
  input_size: 3
  output_size: 1
  hidden_layers: 4
  layer_sizes: [8, 16, 8, 4]
  activation: "relu"
  output_activation: "tanh"
  use_bias: false
```

In [ ]:
class ConfigurableNN(nn.Module):
    
    def __init__(self, config_path="config.yaml"):
        super(ConfigurableNN, self).__init__()
        
        self.config = self.load_config(config_path)
        network_config = self.config['network']
        
        self.input_size = network_config['input_size']
        self.output_size = network_config['output_size']
        self.hidden_layers = network_config['hidden_layers']
        self.layer_sizes = network_config['layer_sizes']
        self.use_bias = network_config['use_bias']
        
        self.activation = self.get_activation(network_config['activation'])
        self.output_activation = self.get_activation(network_config['output_activation'])
        
        self.layers = nn.ModuleList()
        self.build_layers()
    
    def load_config(self, config_path):
        if not Path(config_path).exists():
            raise FileNotFoundError(f"Konfigurációs fájl nem található: {config_path}")
        
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
    
    def get_activation(self, activation_name):
        activations = {
            'relu': nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh': nn.Tanh(),
            'softplus' : nn.Softplus(),
            'leaky_relu' : nn.LeakyReLU(),
            'none': nn.Identity()
        }
        return activations.get(activation_name.lower(), nn.ReLU())
    
    def build_layers(self):

        # Input
        if len(self.layer_sizes) < self.hidden_layers:
            # If not enough sizes are provided, extend with the last value
            self.layer_sizes = self.layer_sizes + [self.layer_sizes[-1]] * (self.hidden_layers - len(self.layer_sizes))
        
        # First layer
        self.layers.append(nn.Linear(self.input_size, self.layer_sizes[0], bias=self.use_bias))
        
        for i in range(self.hidden_layers - 1):
            self.layers.append(nn.Linear(self.layer_sizes[i], self.layer_sizes[i+1], bias=self.use_bias))
        
        # Output layer
        self.layers.append(nn.Linear(self.layer_sizes[-1], self.output_size, bias=self.use_bias))
    
    def forward(self, x):

        for i in range(len(self.layers) - 1):
            x = self.activation(self.layers[i](x))
        x = self.output_activation(self.layers[-1](x))
        return x
    
    def classify_image(self, image_path, output_path="output_heatmap.png", cmap_name="inferno"):

        self.eval()

        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        img_array = (img_array - 127.5) / 127.5

        height, width, _ = img_array.shape
        output_array = np.zeros((height, width), dtype=np.float32)

        with torch.no_grad():
            for i in range(height):
                for j in range(width):
                    pixel = torch.tensor(img_array[i, j], dtype=torch.float32).unsqueeze(0).to("cuda")
                    output = self.forward(pixel).item()
                    output_array[i, j] = output
        
        output_array = np.clip(output_array, 0.0, 1.0)

        cmap = cm.get_cmap(cmap_name)
        heatmap = cmap(output_array)[:, :, :3]

        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))
        heatmap_img.save(output_path)

        print(f"Hőtérkép elmentve ide: {output_path}")

        return output_array

In [ ]:
configurable_model = ConfigurableNN(config_path="config.yaml")
configurable_model.to("cuda") if torch.cuda.is_available() else configurable_model.to("cpu")
configurable_model = train_model(configurable_model, X_train, y_train, X_val, y_val)
test(configurable_model.to("cpu"), X_test, y_test)

## Torch Quantizer

I proceed in the same way as above. It is important to create the network with the same parameters as in the FP32 model because of copying the weights.

In [ ]:
class ConfigurableQuantizedNN(nn.Module):
    def __init__(self, config_path="config.yaml"):
        super(ConfigurableQuantizedNN, self).__init__()
        
        self.config = self.load_config(config_path)
        network_config = self.config['network']
        
        self.input_size = network_config['input_size']
        self.output_size = network_config['output_size']
        self.hidden_layers = network_config['hidden_layers']
        self.layer_sizes = network_config['layer_sizes']
        self.use_bias = network_config['use_bias']
        
        self.activation = self.get_activation(network_config['activation'])
        self.output_activation = self.get_activation(network_config['output_activation'])

        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        
        self.layers = nn.ModuleList()
        self.build_layers()
    
    def load_config(self, config_path):
        if not Path(config_path).exists():
            raise FileNotFoundError(f"Konfigurációs fájl nem található: {config_path}")
        
        with open(config_path, 'r') as f:
            return yaml.safe_load(f)
    
    def get_activation(self, activation_name):
        activations = {
            'relu': nn.ReLU(),
            'sigmoid': nn.Sigmoid(),
            'tanh': nn.Tanh(),
            'leaky_relu': nn.LeakyReLU(),
            'softplus' : nn.Softplus(),
            'none': nn.Identity()
        }
        return activations.get(activation_name.lower(), nn.ReLU())
    
    def build_layers(self):

        # Input
        if len(self.layer_sizes) < self.hidden_layers:
            # If not enough sizes are provided, extend with the last value
            self.layer_sizes = self.layer_sizes + [self.layer_sizes[-1]] * (self.hidden_layers - len(self.layer_sizes))
        
        # First layer
        self.layers.append(nn.Linear(self.input_size, self.layer_sizes[0], bias=self.use_bias))
        
        for i in range(self.hidden_layers - 1):
            self.layers.append(nn.Linear(self.layer_sizes[i], self.layer_sizes[i+1], bias=self.use_bias))
        
        # Output layer
        self.layers.append(nn.Linear(self.layer_sizes[-1], self.output_size, bias=self.use_bias))
        
    def forward(self, x):
        x = self.quant(x)
        for i in range(len(self.layers) - 1):
            x = self.activation(self.layers[i](x))
        x = self.output_activation(self.layers[-1](x))
        x = self.dequant(x)
        return x

configurable_quant_model = ConfigurableQuantizedNN(config_path="config.yaml").to(device="cpu")
configurable_quant_model.load_state_dict(configurable_model.state_dict())
configurable_quant_model.eval()

configurable_quant_model.qconfig = torch.ao.quantization.default_qconfig
quantconfigurable_quant_model_model = torch.ao.quantization.prepare(configurable_quant_model, inplace=True)
# Test data for quantization
test(configurable_quant_model, X_test, y_test)
configurable_quant_model = torch.ao.quantization.convert(configurable_quant_model, inplace=True)
# Testing the quantized model
test(configurable_quant_model, X_test, y_test)


## Brevitas Quantizer

In [ ]:
class ConfigurableQuantNN_Brevitas(nn.Module):
    def __init__(self, config_path="config.yaml"):
        super().__init__()

        self.config = self.load_config(config_path)
        network_config = self.config["network"]
        quant_config = self.config.get("quantization", {})

        self.input_size = network_config["input_size"]
        self.output_size = network_config["output_size"]
        self.hidden_layers = network_config["hidden_layers"]
        self.layer_sizes = network_config["layer_sizes"]
        self.use_bias = network_config["use_bias"]

        self.act_quant_type = quant_config.get("act_quant", "Int8ActPerTensorFloat")
        self.weight_quant_type = quant_config.get("weight_quant", "Int8WeightPerTensorFloat")

        self.act_quant = self.get_act_quantizer(self.act_quant_type)
        self.weight_quant = self.get_weight_quantizer(self.weight_quant_type)

        self.activation_name = network_config["activation"]
        self.output_activation_name = network_config["output_activation"]

        self.layers = nn.ModuleList()
        self.build_layers()

    def load_config(self, config_path):
        if not Path(config_path).exists():
            raise FileNotFoundError(f"Konfig fájl nem található: {config_path}")
        with open(config_path, "r") as f:
            return yaml.safe_load(f)

    
    def get_act_quantizer(self, name):
        mapping = {
            "Int8ActPerTensorFloat": Int8ActPerTensorFloat,
            "SignedTernaryActPerTensorConst": SignedTernaryActPerTensorConst, 
            "Int8ActPerTensorFixedPoint" : Int8ActPerTensorFixedPoint,
        }
        return mapping[name]

    def get_weight_quantizer(self, name):
        mapping = {
            "Int8WeightPerTensorFloat": Int8WeightPerTensorFloat,
            "SignedTernaryWeightPerTensorConst": SignedTernaryWeightPerTensorConst,
            'Int8WeightPerTensorFixedPoint' : Int8WeightPerTensorFixedPoint,
        }
        return mapping[name]

    def get_activation(self, activation_name):
        activations = {
            "relu": lambda: qnn.QuantReLU(act_quant=self.act_quant),
            "sigmoid": lambda: qnn.QuantSigmoid(act_quant=self.act_quant, signed=False),
            "tanh": lambda: qnn.QuantTanh(act_quant=self.act_quant),
            "leaky_relu" : lambda: nn.LeakyReLU(),
            "softplus" : lambda: nn.Softplus(),
            "none": lambda: qnn.QuantIdentity(act_quant=self.act_quant),
        }
        return activations.get(activation_name.lower(), lambda: qnn.QuantReLU(act_quant=self.act_quant))()

    def build_layers(self):
        if len(self.layer_sizes) < self.hidden_layers:
            self.layer_sizes = self.layer_sizes + [self.layer_sizes[-1]] * (self.hidden_layers - len(self.layer_sizes))

        self.layers.append(qnn.QuantIdentity(act_quant=self.act_quant, return_quant_tensor=True))

        self.layers.append(
            qnn.QuantLinear(
                in_features=self.input_size,
                out_features=self.layer_sizes[0],
                weight_quant=self.weight_quant,
                bias=self.use_bias,
                bias_quant=Int8BiasPerTensorFixedPointInternalScaling,
            )
        )
        self.layers.append(self.get_activation(self.activation_name))

        for i in range(self.hidden_layers - 1):
            self.layers.append(
                qnn.QuantLinear(
                    in_features=self.layer_sizes[i],
                    out_features=self.layer_sizes[i + 1],
                    weight_quant=self.weight_quant,
                    bias=self.use_bias,
                    bias_quant=Int8BiasPerTensorFixedPointInternalScaling,
                )
            )
            self.layers.append(self.get_activation(self.activation_name))

        self.layers.append(
            qnn.QuantLinear(
                in_features=self.layer_sizes[-1],
                out_features=self.output_size,
                weight_quant=self.weight_quant,
                bias=self.use_bias,
                bias_quant=Int8BiasPerTensorFixedPointInternalScaling,
            )
        )
        self.layers.append(self.get_activation(self.output_activation_name))

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x*2
    
    def classify_image(self, image_path, output_path="output_heatmap.png", cmap_name="inferno"):

        self.eval()

        img = Image.open(image_path).convert('RGB')
        img_array = np.array(img, dtype=np.float32)
        img_array = (img_array - 127.5) / 127.5

        height, width, _ = img_array.shape
        output_array = np.zeros((height, width), dtype=np.float32)

        with torch.no_grad():
            for i in range(height):
                for j in range(width):
                    pixel = torch.tensor(img_array[i, j], dtype=torch.float32).unsqueeze(0).to("cuda")
                    output = self.forward(pixel).item()
                    output_array[i, j] = output
        
        cmap = cm.get_cmap(cmap_name)
        heatmap = cmap(output_array)[:, :, :3]

        heatmap_img = Image.fromarray((heatmap * 255).astype(np.uint8))
        heatmap_img.save(output_path)

        print(f"Hőtérkép elmentve ide: {output_path}")

        return output_array

configurable_brevitas_quant_model = ConfigurableQuantNN_Brevitas(config_path="config.yaml").to(device="cpu")

In [ ]:
def copy_weights_from_fp(fp_model, brevitas_model):
    
    fp_linears = [m for m in fp_model.layers if isinstance(m, nn.Linear)]
    brevitas_linears = [m for m in brevitas_model.layers if isinstance(m, qnn.QuantLinear)]

    if len(fp_linears) != len(brevitas_linears):
        raise ValueError(f"Rétegek száma nem egyezik! FP={len(fp_linears)}, Brevitas={len(brevitas_linears)}")

    with torch.no_grad():
        for l_fp, l_q in zip(fp_linears, brevitas_linears):
            l_q.weight.data.copy_(l_fp.weight.data)
            if l_fp.bias is not None and l_q.bias is not None:
                l_q.bias.data.copy_(l_fp.bias.data)

copy_weights_from_fp(configurable_model, configurable_brevitas_quant_model)
configurable_brevitas_quant_model.eval()

calibration_data = X_train[:10000] 
test(configurable_brevitas_quant_model, X_test, y_test)

configurable_brevitas_quant_model.train()
with torch.no_grad():
    for i in range(min(100, len(calibration_data))):
        sample = calibration_data[i].unsqueeze(0)
        _ = configurable_brevitas_quant_model(sample)
        if i % 20 == 0:
            print(f"Kalibráció: {i}/100")
            
configurable_brevitas_quant_model.eval()
test(configurable_brevitas_quant_model, X_test, y_test)

Saving weights to files

In [ ]:
brevitas_linears = [m for m in configurable_brevitas_quant_model.layers if isinstance(m, qnn.QuantLinear)]

extract_params_hls(
    brevitas_linears,
    "brevitas_conf_params",
    backend="brevitas"
)


Creating and saving the test dataset

In [ ]:
def save_dataset(X, y, filename_prefix="dataset"):
    X_scaled = (X * 256)
    y = y.numpy()

    np.savetxt(f"{filename_prefix}_X.txt", X_scaled.numpy(), fmt="%d", delimiter=", ")
    np.savetxt(f"{filename_prefix}_y.txt", y, fmt="%d", delimiter=", ")
    print(f"Dataset saved to {filename_prefix}_X.txt and {filename_prefix}_y.txt")
save_dataset(X_test, y_test, filename_prefix="val_dataset")

## Modeling the Quantized Forward Pass

As a reference, I collect the network parameters here.

In [ ]:
brevitas_linears = [m for m in configurable_brevitas_quant_model.layers if isinstance(m, qnn.QuantLinear)]
for layer in brevitas_linears:
    print("Weights:")
    print(layer.quant_weight().int())
    print(layer.quant_weight())
    print("Bias:")
    print(layer.quant_bias().int() if hasattr(layer.quant_bias(), "int") else layer.quant_bias())
    print(layer.quant_bias())
    print("\n")



This code runs two forward passes in parallel: in one of them I use floating-point operations, and in the other I use integer operations. The idea was to use static scaling factors and bit widths, but due to the loss of precision that occurs in the activations, I also need to handle this dynamically.

In [ ]:
out1 = []
out2 = []

actfp = nn.Softplus()

beta=1/(256*64) # beta = 1/(input_scale*weight_scale)
actint = nn.Softplus(beta=beta)

beta2 = 1/(32*64)
actint2 = nn.Softplus(beta=beta2)

print(f"adat: {X_test[0]*256}")

for i in range(4):
    acc1 = 0
    acc2 = 0
    for j in range(3):
        acc1 += (configurable_brevitas_quant_model.layers[1].weight.data[i][j] * X_test[0][j])                       
        acc2 += (configurable_brevitas_quant_model.layers[1].weight.data[i][j]*64 * (X_test[0][j]*256)) # 64 = 2^6 súly aktiváció scale; 256 = 2^8 input sclae
        print(f"accumulator: {acc2}")
    out1.append(actfp(acc1 + configurable_brevitas_quant_model.layers[1].bias.data[i])) 
    x = (acc2 + configurable_brevitas_quant_model.layers[1].bias.data[i] * 64 * 256) # bias a réteg együttes scale szerint
    print(f"bias után: {x}")
    x = actint(x) # fent definiált aktivációs fv szintén a scale szerint
    print(f"act után: {x}")
    print(x/(256*64)) # ellenőrzésképpen, hogy egyezik-e a fp értékkel visszaskálázom
    x = x / (512) # ezt a lépés határozza meg a kimenet scaling faktorát, ezt szeretném dinamikussá tenni, egy priority encoderrel gondoltam megoldani, out_sf = 14 - 9 = 5
    out2.append(x)

print(out1)
print(out2)
print('\n')


layer2out1 = []
layer2out2 = []

for i in range(4):
    acc1 = 0
    acc2 = 0
    for j in range(4):
        acc1 += (configurable_brevitas_quant_model.layers[3].weight.data[i][j] * out1[j])                      
        acc2 += (configurable_brevitas_quant_model.layers[3].weight.data[i][j]*64 * (out2[j])) # az out2 hozza magával az előző réteg kimenetének a sf-jét 2^5=32
        print(f"accumulator: {acc2}")
    layer2out1.append(actfp(acc1 + configurable_brevitas_quant_model.layers[3].bias.data[i]))
    x = acc2 + configurable_brevitas_quant_model.layers[3].bias.data[i] * 32 * 64
    print(f"bias után: {x}")
    x = actint2(x)
    print(f"act után: {x}")
    print(x/(32*64))
    layer2out2.append(x/64) # out_sf = 11 - 7 = 4


print(layer2out1)
print(layer2out2)

layer3out1 = []
layer3out2 = []


outfp = nn.Sigmoid()

print('\n')


def outint(x):
    beta = 16*64
    return beta/(1 + torch.exp(-x/beta)) 

for i in range(1):
    acc1 = 0
    acc2 = 0
    for j in range(4):
        acc1 += (configurable_brevitas_quant_model.layers[5].weight.data[i][j] * layer2out1[j])                      
        acc2 += (configurable_brevitas_quant_model.layers[5].weight.data[i][j]*64 * (layer2out2[j]))
    layer3out1.append(outfp(acc1 + configurable_brevitas_quant_model.layers[5].bias.data[i]))
    x = (acc2 + configurable_brevitas_quant_model.layers[5].bias.data[i] * 16 * 64)
    x = outint(x)
    print(x / (16*64))
    layer3out2.append(x)

print(layer3out1)
print(layer3out2)